In [54]:
INSTRUCTIONS = """
You are Revenue AI Copilot, an assistant specialized in hotel revenue management.

Answer questions using only the provided context.

If the answer is not found in the context, respond:
'I don't know based on the available documents.'
"""

In [55]:
# Setup
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def llm(user_prompt, model="llama-3.1-8b-instant"):
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [56]:
from pathlib import Path
from pypdf import PdfReader

pdf_dir = Path("data/raw")
pdf_files = list(pdf_dir.glob("*.pdf"))

pdf_files

[PosixPath('data/raw/Revenue-Management-Manual-Xotels-2.pdf'),
 PosixPath('data/raw/ebook-bi-time-saving-toolkit-for-revenue-managers-0424.pdf'),
 PosixPath('data/raw/2019-hsmai-and-sit-revenue-management-metrics-study-final.pdf'),
 PosixPath('data/raw/Hotel Revenue Guide eBook_18.07.2023.pdf'),
 PosixPath('data/raw/Beginners_Guide_to_Revenue_Management.pdf')]

In [57]:
documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            documents.append({
                "source": pdf_file.name,
                "page": page_number + 1,
                "text": text
            })

len(documents)

183

In [58]:
from pypdf import PdfReader

documents = []

for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            documents.append({
                "source": pdf_file.name,
                "page": page_number + 1,
                "text": text
            })

print(f"Loaded {len(documents)} pages")

Loaded 183 pages


In [59]:
documents[0]

{'source': 'Revenue-Management-Manual-Xotels-2.pdf',
 'page': 1,
 'text': ' \n \n \n  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n  \n \nLeadership in Revenue Management \n0\n20\n40\n60\n80\n100\n120\n-1 0 1 2 3 4 5 6 7 8 9 10 15 20 30 40 50 60 70 80\nRooms sold\nDays before D Day\nHistorical booking curve'}

In [60]:
def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk.strip())

        start = end - overlap

    return chunks

In [61]:
chunks = []

for doc in documents:
    text = doc["text"]

    for chunk_id, chunk in enumerate(chunk_text(text)):
        if len(chunk) > 100:
            chunks.append({
                "source": doc["source"],
                "page": doc["page"],
                "chunk_id": chunk_id,
                "text": chunk
            })

len(chunks)

404

In [62]:
chunks[0]

{'source': 'Revenue-Management-Manual-Xotels-2.pdf',
 'page': 1,
 'chunk_id': 0,
 'text': 'Leadership in Revenue Management \n0\n20\n40\n60\n80\n100\n120\n-1 0 1 2 3 4 5 6 7 8 9 10 15 20 30 40 50 60 70 80\nRooms sold\nDays before D Day\nHistorical booking curve'}

In [63]:
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [64]:
clean_chunks = []

for chunk in chunks:
    text = clean_text(chunk["text"])

    if len(text) > 200:
        clean_chunks.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "text": text
        })

len(clean_chunks)

371

In [65]:
clean_chunks[5]

{'source': 'Revenue-Management-Manual-Xotels-2.pdf',
 'page': 2,
 'chunk_id': 5,
 'text': '.......... 27 Forecasting Model and Tool ........................................................................................................... 27 ACTION 7 ............................................................................................................................................ 29'}

In [66]:
def search(query, top_k=5):
    results = []

    for chunk in clean_chunks:
        text = chunk["text"].lower()
        score = 0

        for word in query.lower().split():
            if word in text:
                score += 1

        if score > 0:
            results.append((score, chunk))

    results = sorted(results, reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in results[:top_k]]

In [67]:
search("What is revenue management?")

[{'source': 'Revenue-Management-Manual-Xotels-2.pdf',
  'page': 5,
  'chunk_id': 1,
  'text': 'ation about the market so that you can be proactive and not reactive. Use the information to divide your market and adjust your products through distribution, to the right customer at the right time and at the right price. Revenue Management is not only maximizing in high period demand, it helps stimulating demand in low periods while avoiding pricing cannibalism. Revenue Management is long term strategic, takes all revenue with their profitability into consideration, can sell low rates even in high demand period. What makes hotels suitable to be able to apply Revenue Management? \uf0b7 fixed capacity \uf0b7 perishable product \uf0b7 high fixed costs and low variable costs \uf0b7 product can be priced differently \uf0b7 demand evolves \uf0b7 product can be sold in advance \uf0b7 market can be segmented Rev'},
 {'source': 'Revenue-Management-Manual-Xotels-2.pdf',
  'page': 7,
  'chunk_id': 0,


In [68]:
def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += f"""
Source: {doc["source"]}
Page: {doc["page"]}
Text: {doc["text"]}
"""

    prompt = f"""
You are Revenue AI Copilot, an assistant specialized in hotel revenue management.

Answer the QUESTION using only the CONTEXT.
If the answer is not in the CONTEXT, say:
"I don't know based on the available documents."

QUESTION:
{query}

CONTEXT:
{context}
"""
    return prompt

In [69]:
def rag(query, top_k=10):
    search_results = search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [70]:
rag("What is hotel revenue management?")

'Hotel revenue management refers to the process of maximizing revenue and profits through the strategic management of pricing, hotel inventory, and distribution channels.'

In [71]:
rag("What are the ingredients of effective hotel revenue management?")

'The ingredients of effective hotel revenue management are:\n\n1. Market Segmentation\n2. Historical Demand and Booking Patterns\n3. Demand Forecast and Displacement Analysis\n4. Pricing and Inventory Management\n5. Overbooking\n6. Information Systems\n\nThese ingredients are mentioned in the Revenue-Management-Manual-Xotels-2.pdf document on page 7.'

In [72]:
rag("What KPIs are used in hotel revenue management?")

'Some key performance indicators (KPIs) used in hotel revenue management are:\n\n1. Occupancy\n2. Average Daily Rate (ADR)\n3. Revenue Per Available Room (RevPAR)\n4. Day of week analysis (by account)\n5. Market segmentation by traveler type (leisure, business, bleisure, group business)\n6. Pricing strategies (hotel pricing optimization, dynamic pricing, differentiated pricing)\n7. Overbooking\n8. Information Systems\n9. Market data (occupancy, ADR, RevPAR, etc.)\n10. Distribution strategies (channel management, rate parity, etc.)\n\nThese KPIs help revenue managers analyze market demand, adjust pricing and inventory levels, and employ various techniques to optimize revenue and profit margins.'

In [73]:
rag("What is the role of pricing and inventory management?")

'The role of pricing and inventory management is to maximize revenue and profits through the strategic management of pricing, hotel inventory, and distribution channels. \n\nAccording to the Hotel Revenue Guide eBook_18.07.2023.pdf, page 5, "Revenue managers analyze market demand, adjust pricing and inventory levels, and employ various techniques to optimize revenue and profit margins." \n\nAdditionally, the Hotel Revenue Guide eBook_18.07.2023.pdf, page 22, states that "Technology provides a range of tools and platforms that help hotels maximize their revenue potential" and that "Revenue management involves collecting and analyzing large volumes of data to make informed decisions." \n\nTherefore, pricing and inventory management are crucial components of hotel revenue management, as they enable hotels to optimize their pricing and inventory levels to maximize revenue and profits.'

In [74]:
import minsearch

In [75]:
index = minsearch.Index(
    text_fields=["text"],
    keyword_fields=["source", "page"]
)

In [76]:
index.fit(clean_chunks)

In [77]:
def search(query, top_k=5):
    boost = {
        "text": 1.0
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=top_k
    )

    return results

In [78]:
rag("What KPIs are used in hotel revenue management?")

'Some key performance indicators (KPIs) used in hotel revenue management are:\n\n1. Average Daily Rate (ADR)\n2. Occupancy Rate\n3. Revenue Per Available Room (RevPAR)'

In [79]:
rag("What is ADR?")

'Average Daily Rate (ADR) is the average price charged by the hotel for a room night. It is calculated as total room revenue divided by rooms sold.'

In [80]:
rag("What is RevPAR?")

'RevPAR stands for Revenue Per Available Room. It is a metric that combines both occupancy and Average Daily Rate (ADR) into one statistic, providing a more accurate view of hotel performance.'

In [81]:
rag("Why is market segmentation important?")

'Market segmentation is important because it allows you to target a variety of consumer groups with different behavior with an offer that matches their needs and budget level. It helps to identify the purpose of the trip: either business or leisure, and clear distinction must also be achieved between individual and group business. Market segmentation shall help you identify the trends of your business, such as Length of Stay, Day of Weeks stays, Total Revenue per room, Total Revenue per client, etc.'

In [82]:
rag("What are the benefits of dynamic pricing?")

"The benefits of dynamic pricing are:\n\n1. Maximizing revenue: By adjusting prices in real-time based on demand and other factors, hotels can increase revenue while ensuring that guests perceive the value of their stay.\n2. Increasing profits: Dynamic pricing helps hotels capitalize on high-demand periods and attract price-sensitive customers during low-demand periods.\n3. Flexibility: Dynamic pricing involves adjusting room rates in real-time, allowing hotels to respond quickly to changes in demand and market conditions.\n4. Differentiation: By offering competitive pricing, hotels can differentiate themselves from competitors and attract price-sensitive customers.\n5. Maximizing occupancy: Dynamic pricing helps hotels to maximize occupancy by adjusting prices to match demand.\n6. Increasing ADR: Dynamic pricing helps hotels to increase Average Daily Rate (ADR) by adjusting prices to match demand.\n7. Automating pricing decisions: Tools like Cloudbeds' Pricing Intelligence Engine (PIE

In [83]:
rag("What is channel management?")

"Channel management refers to the process of managing a hotel's distribution channels and rates across multiple channels simultaneously, such as direct bookings and online travel agencies. This involves ensuring that inventory is profitably distributed across various booking channels to maximize visibility and revenue, while maintaining rate parity to avoid customer confusion and dissatisfaction."

In [84]:
test_questions = [
    "What is hotel revenue management?",
    "What KPIs are used in hotel revenue management?",
    "Why is market segmentation important?",
    "What are the benefits of dynamic pricing?",
    "What is channel management?"
]

for q in test_questions:
    print("QUESTION:", q)
    print("ANSWER:", rag(q))
    print("-" * 80)

QUESTION: What is hotel revenue management?
ANSWER: Hotel revenue management refers to the process of maximizing revenue and profits through the strategic management of pricing, hotel inventory, and distribution channels. Revenue managers analyze market demand, adjust pricing and inventory levels, and employ various techniques to optimize revenue and profit margins. By optimizing these factors, hoteliers can attract more guests, increase occupancy rates, and drive revenue growth. The goal of hotel revenue management is to achieve the right balance between occupancy and rate, ensuring that hotels are operating at maximum capacity while also charging optimal prices.
--------------------------------------------------------------------------------
QUESTION: What KPIs are used in hotel revenue management?
ANSWER: Some key performance indicators (KPIs) used in hotel revenue management are:

1. Average Daily Rate (ADR)
2. Occupancy Rate
3. Revenue Per Available Room (RevPAR)
-----------------

In [85]:
rag("What is RevPAR?")

'RevPAR stands for Revenue Per Available Room. It is a metric that combines both occupancy and Average Daily Rate (ADR) into one statistic, providing a more accurate view of hotel performance.'

In [86]:
rag("How can hotels increase revenue during low demand periods?")

'To increase revenue during low demand periods, hotels can use various strategies mentioned in the provided context. \n\n1. Dynamic pricing: Adjust room rates in real-time based on factors like demand, competition, and market conditions to attract price-sensitive customers during low-demand periods.\n2. Stay restrictions: Implement minimum length of stay (MinLOS) restrictions to filter less profitable clients during peak demand seasons, thus increasing the resulting room revenue.\n3. Analyze and adjust room type differentials: Monitor and analyze the booking pace of different room types during different demand seasons and increase or decrease the difference in rates between them to maximize resulting revenue.\n4. Use technology: Leverage revenue management software and tools to collect, process, and analyze data more efficiently, enabling hotels to make better decisions and improve their revenue.\n5. Forecasting and pricing: Use advanced algorithms to analyze historical data, competito